# GR00T N1.7 on an OXE dataset: dataset info and the model adapter

`robotdataset` returns TED batches: flat TensorDicts keyed by `"/"`-separated paths. It does not know which leaf
is which model input, in what order, or where a group sits inside an assembled state or action vector. Two layers
close that gap without touching any dataset metadata files:

```
loader ──▶ DatasetInfo ──▶ viola_gr00t.yaml ──(you edit it)──▶ Gr00tAdapter ──▶ VLAStepData, GR00T configs, statistics
          (robotdataset.info)                                  (robotdataset.adapters.gr00t)
```

This notebook walks that path for the Open X-Embodiment dataset `viola` and stops at the model inputs of
`Gr00tN1d7Processor`; it needs neither GPU nor model weights. The training loop that consumes the same objects is
`finetune_scratch.ipynb` in the `finetune-groot` repository (reference docs: `doc/dataset_info.md`, `doc/adapters.md`).

Requirements: `robotdataset[oxe]` and the `gr00t` package (Isaac-GR00T N1.7). The first run streams five episodes
from `gs://gresearch/robotics/viola` into the robotdataset cache; later runs are offline.

In [1]:
from dataclasses import replace
from pathlib import Path

import numpy as np

from robotdataset.adapters.gr00t import Gr00tAdapter
from robotdataset.info import DatasetInfo

TAG = "new_embodiment"  # GR00T's embodiment tag for a custom robot
TRAIN_EPISODES, VAL_EPISODES = [0, 1, 2, 3], [4]
YAML_PATH = Path("viola_gr00t.yaml")

## 1. List what the dataset provides

`DatasetInfo.from_loader` builds the loader on a few episodes and describes every TED leaf: path, kind, dtype and
per-step shape (images are HWC as stored). TED bookkeeping (`next/...`, `done`, ...) is hidden. The same listing is
available from a shell: `python -m robotdataset.info --loader OXEDataset --kwargs '{"dataset_name": "viola"}' --episodes 0:4 --adapter gr00t -o viola_gr00t.yaml`.

In [2]:
info = DatasetInfo.from_loader("OXEDataset", {"dataset_name": "viola"}, episodes=TRAIN_EPISODES)
print(info.table())

OXEDataset: 4 episode(s), 1865 step(s)
path                                      kind    dtype    shape          use  name
----------------------------------------  ------  -------  -------------  ---  ----------------------------
action/gripper_closedness_action          action  float32  []             no   gripper_closedness_action
action/terminate_episode                  action  float32  []             no   terminate_episode
action/rotation_delta                     action  float32  [3]            no   rotation_delta
action/world_vector                       action  float32  [3]            no   world_vector
observation/agentview_rgb                 image   uint8    [224, 224, 3]  no   agentview_rgb
observation/natural_language_embedding    text    float32  [512]          no   natural_language_embedding
observation/joint_states                  state   float32  [7]            no   joint_states
observation/natural_language_instruction  text    string   []             no   natural_lan

## 2. Choose and order the modalities

The listing is also a YAML file: `use: true` marks what the model gets, and the order of the list is the order
handed to the model. `name` is the GR00T-side group name, `split` can cut one vector leaf into several groups, and
the `gr00t:` block holds the GR00T settings (action horizon, windows, action representation). You would open the
file and edit it by hand; the cell below makes the same edit in code so that the notebook runs unattended.

For `viola`: two cameras, joint and gripper state, the language instruction, and the three action leaves. Its actions
are per-step delta commands, so they stay `ABSOLUTE` (the default): the model predicts them as they are.
`terminate_episode`, `ee_states` and the language embedding are left out.

In [3]:
PICK = [
    "observation/agentview_rgb", "observation/eye_in_hand_rgb",          # video
    "observation/joint_states", "observation/gripper_states",            # state
    "observation/natural_language_instruction",                          # language
    "action/world_vector", "action/rotation_delta", "action/gripper_closedness_action",  # action
]
chosen = [replace(info[p], use=True) for p in PICK]
info = replace(info, modalities=tuple(chosen + [m for m in info.modalities if m.path not in PICK]))
info = Gr00tAdapter.with_defaults(info)  # adds the gr00t: block with its defaults (horizon 16, windows [0])
info.to_yaml(str(YAML_PATH))
print(YAML_PATH.read_text())

# robotdataset dataset info -- generated; edit the modalities, then load the file in a model adapter.
#   use:   true on every modality the model should get (all start as false)
#   order: the order of this list is the order handed to the model
#   name:  the model-side group name (defaults to the shortest unique path suffix)
#   role:  optional override of the role implied by kind (video | state | action | language)
#   split: optional, cuts one vector leaf into named groups: [{name: eef, columns: [0, 6]}, ...]
# kind/dtype/shape describe the data; they are checked against the loader when the file is loaded.
dataset:
  loader: OXEDataset
  kwargs: {dataset_name: viola}
  episodes_loaded: 4
  steps_loaded: 1865
modalities:
  - {path: observation/agentview_rgb, kind: image, dtype: uint8, shape: [224, 224, 3], use: true, name: agentview_rgb}
  - {path: observation/eye_in_hand_rgb, kind: image, dtype: uint8, shape: [224, 224, 3], use: true, name: eye_in_hand_rgb}
  - {path: observation/jo

## 3. Load the file in the model adapter

`Gr00tAdapter.from_yaml` reads the edited file. Nothing has touched the data yet. `index_map()` answers the question
robotdataset itself does not track: which columns of which leaf end up where in the assembled state and action
vectors (groups concatenated in file order). It is also how a predicted action vector splits back into groups.

In [4]:
adapter = Gr00tAdapter.from_yaml(str(YAML_PATH))
for role, entries in adapter.index_map().items():
    print(f"{role} vector:")
    for e in entries:
        print(f"  {e['group']:<28} <- {e['path']}[{e['columns'][0]}:{e['columns'][1]}]  at {tuple(e['assembled'])}")
modality_configs = adapter.modality_configs()  # GR00T ModalityConfig per modality, ready for the processor
{k: (list(v.modality_keys), list(v.delta_indices)) for k, v in modality_configs.items()}

state vector:
  joint_states                 <- observation/joint_states[0:7]  at (0, 7)
  gripper_states               <- observation/gripper_states[0:1]  at (7, 8)
action vector:
  world_vector                 <- action/world_vector[0:3]  at (0, 3)
  rotation_delta               <- action/rotation_delta[0:3]  at (3, 6)
  gripper_closedness_action    <- action/gripper_closedness_action[0:1]  at (6, 7)


{'video': (['agentview_rgb', 'eye_in_hand_rgb'], [0]),
 'state': (['joint_states', 'gripper_states'], [0]),
 'action': (['world_vector', 'rotation_delta', 'gripper_closedness_action'],
  [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15]),
 'language': (['annotation.human.task_description'], [0])}

## 4. Build the loaders and look at a batch

`build_dataset` instantiates the loader named in the file with the sampling windows the selection needs (one frame
for video and state, a 16-step window for action) and checks the file against a fresh listing of the loader and a
sampled batch. A stale file fails with one message naming everything that changed. `to_steps` cuts the sampled
leaves into the selected groups and wraps each sample as GR00T's `VLAStepData`.

In [5]:
train_ds = adapter.build_dataset(episodes=TRAIN_EPISODES, batch_size=4)
val_ds = adapter.build_dataset(episodes=VAL_EPISODES, batch_size=4)
print("train:", train_ds.num_episodes, "episodes,", len(train_ds), "steps | val:", val_ds.num_episodes, "episodes,", len(val_ds), "steps")

batch = train_ds.sample()
print({m.path: tuple(batch[m.path].shape) for m in adapter.info.selected if not m.is_string})  # (B, T, ...) per TED leaf

step = adapter.to_steps(batch, TAG)[0]
print("text   :", step.text)
print("images :", {k: (np.asarray(v[0]).shape, str(np.asarray(v[0]).dtype)) for k, v in step.images.items()})
print("states :", {k: v.shape for k, v in step.states.items()})
print("actions:", {k: v.shape for k, v in step.actions.items()})

train: 4 episodes, 1865 steps | val: 1 episodes, 719 steps


{'observation/agentview_rgb': (4, 1, 3, 224, 224), 'observation/eye_in_hand_rgb': (4, 1, 3, 224, 224), 'observation/joint_states': (4, 1, 7), 'observation/gripper_states': (4, 1, 1), 'action/world_vector': (4, 16, 3), 'action/rotation_delta': (4, 16, 3), 'action/gripper_closedness_action': (4, 16)}
text   : make coffee
images : {'agentview_rgb': ((224, 224, 3), 'uint8'), 'eye_in_hand_rgb': ((224, 224, 3), 'uint8')}
states : {'joint_states': (1, 7), 'gripper_states': (1, 1)}
actions: {'world_vector': (16, 3), 'rotation_delta': (16, 3), 'gripper_closedness_action': (16, 1)}


## 5. Statistics for the processor

GR00T normalises state and action with per-group statistics. The adapter computes them from the loader you pass:
build that loader from the training episodes only, so held-out episodes do not leak into the normalisation. Nothing is
read from or written to the dataset's metadata files. Quantiles are exact by default because GR00T normalises with
q01/q99; groups declared `RELATIVE` additionally get relative-action statistics (not needed for `viola`).

In [6]:
stats = adapter.statistics(train_ds, TAG)
for role in ("state", "action"):
    for group, s in stats[TAG][role].items():
        print(f"{role:<6} {group:<28} q01 {np.round(s['q01'], 3)}  q99 {np.round(s['q99'], 3)}")

state  joint_states                 q01 [-0.2   -0.19  -0.489 -2.553 -0.145  1.696 -0.005]  q99 [ 0.163  0.719  0.201 -1.611  1.013  2.856  2.803]
state  gripper_states               q01 [0.002]  q99 [0.077]
action world_vector                 q01 [-0.929 -1.    -0.959]  q99 [0.827 0.841 1.   ]
action rotation_delta               q01 [-0.262 -0.237 -0.35 ]  q99 [0.283 0.221 0.34 ]
action gripper_closedness_action    q01 [-1.]  q99 [1.]


## 6. Through the GR00T N1.7 processor

The processor is built exactly as the fine-tuning notebook builds it (here with the defaults of `Gr00tN1d7Config`).
`configure_processor` hands it the statistics and `to_model_batch` goes from a sampled batch to the collated
model inputs, so `model(**inputs)` is the next step in training. Building the processor downloads the tokenizer and
image processor of the Cosmos-Reason2 backbone on first use.

In [7]:
from gr00t.configs.model.gr00t_n1d7 import Gr00tN1d7Config
from gr00t.model.gr00t_n1d7.processing_gr00t_n1d7 import Gr00tN1d7Processor

mc = Gr00tN1d7Config()
processor = Gr00tN1d7Processor(
    modality_configs={TAG: modality_configs},
    use_percentiles=mc.use_percentiles,
    statistics=None,
    embodiment_id_mapping=None,
    image_crop_size=mc.image_crop_size,
    image_target_size=mc.image_target_size,
    random_rotation_angle=mc.random_rotation_angle,
    color_jitter_params=mc.color_jitter_params,
    model_name=mc.model_name,
    model_type=mc.backbone_model_type,
    formalize_language=mc.formalize_language,
    max_state_dim=mc.max_state_dim,
    max_action_dim=mc.max_action_dim,
    apply_sincos_state_encoding=mc.apply_sincos_state_encoding,
    max_action_horizon=mc.action_horizon,
    use_albumentations=mc.use_albumentations_transforms,
    extra_augmentation_config=mc.extra_augmentation_config,
    shortest_image_edge=mc.shortest_image_edge,
    crop_fraction=mc.crop_fraction,
    use_relative_action=mc.use_relative_action,
    exclude_state=mc.exclude_state,
    state_dropout_prob=mc.state_dropout_prob,
    use_mean_std=mc.use_mean_std,
)
assert adapter.action_horizon <= mc.action_horizon
adapter.configure_processor(processor, train_ds, TAG)  # processor.set_statistics(...), computed from train_ds

processor.train()
inputs = adapter.to_model_batch(val_ds.sample(), processor, TAG)


def tensors(x, prefix=""):
    if isinstance(x, dict) or hasattr(x, "items"):
        for k, v in x.items():
            yield from tensors(v, f"{prefix}{k}.")
    elif hasattr(x, "shape"):
        yield prefix.rstrip("."), tuple(x.shape), str(getattr(x, "dtype", ""))


for name, shape, dtype in tensors(inputs):
    print(f"{name:<32} {str(shape):<22} {dtype}")

Overriding statistics for new_embodiment


inputs.input_ids                 (4, 139)               torch.int64
inputs.attention_mask            (4, 139)               torch.int64
inputs.pixel_values              (2048, 1536)           torch.float32
inputs.image_grid_thw            (8, 3)                 torch.int64
inputs.action_mask               (4, 40, 132)           torch.float32
inputs.state                     (4, 1, 132)            torch.float32
inputs.action                    (4, 40, 132)           torch.float32
inputs.embodiment_id             (4,)                   torch.int64


## Your own dataset

Only the first cells change: pick another loader and its arguments (`LeRobotVideoDataset` with `local_path`,
`Table30v2Dataset`, ...), run the listing, edit the YAML. For example, for a LeRobot dataset whose `action` column packs an
end-effector pose and a gripper:

```yaml
modalities:
  - {path: action, kind: action, dtype: float32, shape: [7], use: true, name: action,
     split: [{name: eef, columns: [0, 6]}, {name: gripper, columns: [6, 7]}]}
gr00t:
  action:
    eef: {rep: RELATIVE, type: EEF, format: XYZ_ROTVEC, state_key: cartesian}
```

A `RELATIVE` group needs its reference state (`state_key`) to be a whole state leaf. The two ready-made files for
`finetune-groot` (`stack_tact_v4`, `viola`) are in its `configs/scratch/`.